# Panel Sensitivity Analysis: Perturbation Scenarios and ε* Analysis

This notebook demonstrates the robustness of panel discrimination under perturbations of the similarity matrix.

**Reference:** Based on sensitivity analysis from blackboards/1.md (S01 Study cycle)  
**Date:** 2026-02-15  
**Key Result:** Panel discrimination is structurally robust; requires ε* = 0.351 (35% perturbation) to invert A-B ranking

## Overview

The toy example revealed that Δ_A ≈ Δ_B (gap = 0.021). Is this gap robust to changes in the similarity matrix, or is it an artifact of the particular s_ij values chosen?

We investigate:
1. **Analytical formula** for uniform perturbation: Δ_new = Δ_old - ε(1 - H)
2. **Critical inversion point ε*** where Δ_A = Δ_B
3. **Non-uniform perturbation scenarios** (neighbors closer, uniform shift)
4. **Invariance of S and E** under similarity matrix perturbation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Original data from panel-worked-example
categories = ['C1', 'C2', 'C3', 'C4', 'C5']

# Original similarity matrix
S_orig = np.array([
    [1.00, 0.60, 0.40, 0.10, 0.30],
    [0.60, 1.00, 0.50, 0.15, 0.20],
    [0.40, 0.50, 1.00, 0.35, 0.10],
    [0.10, 0.15, 0.35, 1.00, 0.05],
    [0.30, 0.20, 0.10, 0.05, 1.00]
])

# Publication vectors
pubs_A = np.array([[3,0,2,3,0], [4,1,0,3,0], [2,2,0,3,1], [0,0,3,3,2], [3,0,2,2,1]])
pubs_B = np.array([[5,0,0,0,0], [0,5,0,0,0], [0,0,5,0,0], [0,0,0,5,0], [0,0,0,0,5]])
pubs_C = np.array([[4,3,0,0,0], [5,1,0,0,0], [3,3,1,0,0], [4,2,1,0,0], [5,0,0,0,1]])

# Original exact values (from S03 verification)
Delta_A_orig = 0.559375
Delta_B_orig = 0.580
Delta_C_orig = 0.2456

H_A = 0.258750
H_B = 0.200
H_C = 0.4839

print("Original panel values:")
print(f"Δ_A = {Delta_A_orig:.6f}")
print(f"Δ_B = {Delta_B_orig:.6f}")
print(f"Δ_C = {Delta_C_orig:.6f}")
print(f"\nGap: Δ_A - Δ_B = {Delta_A_orig - Delta_B_orig:.6f}")
print(f"\nHerfindahl indices:")
print(f"H_A = {H_A:.6f}")
print(f"H_B = {H_B:.6f}")
print(f"H_C = {H_C:.6f}")

## 1. Analytical Result: Uniform Perturbation Formula

### Proposition

Under uniform shift $s_{ij} \to s_{ij} + \varepsilon$ for all $i \neq j$ (diagonal unchanged):

$$\Delta_{\text{new}} = \Delta_{\text{old}} - \varepsilon (1 - H)$$

where $H = \sum_i p_i^2$ is the Herfindahl concentration index.

### Proof

$$\Delta = 1 - \sum_{i,j} s_{ij} p_i p_j$$

Under perturbation:
$$\Delta_{\text{new}} = 1 - \sum_{i,j} (s_{ij} + \varepsilon \mathbf{1}_{i \neq j}) p_i p_j$$
$$= \Delta_{\text{old}} - \varepsilon \sum_{i \neq j} p_i p_j$$
$$= \Delta_{\text{old}} - \varepsilon (1 - H) \quad \square$$

### Corollary: Gap Evolution

The difference between any two researchers evolves as:

$$\Delta_{A,\text{new}} - \Delta_{B,\text{new}} = (\Delta_A - \Delta_B) + \varepsilon (H_A - H_B)$$

The gap vanishes at critical perturbation:

$$\varepsilon^* = -\frac{\Delta_A - \Delta_B}{H_A - H_B}$$

In [ ]:
def compute_diversity_from_p(p, S_matrix):
    """
    Compute Rao-Stirling diversity from proportion vector and similarity matrix.
    """
    total_sim = np.dot(p, np.dot(S_matrix, p))
    return 1 - total_sim

def analytical_uniform_perturbation(Delta_old, H, epsilon):
    """
    Apply analytical formula for uniform perturbation.
    
    Δ_new = Δ_old - ε(1 - H)
    """
    return Delta_old - epsilon * (1 - H)

# Compute critical inversion point ε*
gap_orig = Delta_A_orig - Delta_B_orig
H_diff = H_A - H_B

epsilon_star = -gap_orig / H_diff

print("\n" + "="*80)
print("CRITICAL INVERSION POINT ε*")
print("="*80)
print(f"\nOriginal gap: Δ_A - Δ_B = {gap_orig:.6f}")
print(f"H difference: H_A - H_B = {H_diff:.6f}")
print(f"\nCritical perturbation: ε* = -({gap_orig:.6f}) / ({H_diff:.6f})")
print(f"                       ε* = {epsilon_star:.6f}")
print(f"\nInterpretation: Requires {epsilon_star*100:.1f}% perturbation of ALL similarity values")
print("                to invert A-B ranking (make Δ_A = Δ_B)")

# Verify analytical formula at ε*
Delta_A_at_star = analytical_uniform_perturbation(Delta_A_orig, H_A, epsilon_star)
Delta_B_at_star = analytical_uniform_perturbation(Delta_B_orig, H_B, epsilon_star)

print(f"\nVerification at ε* = {epsilon_star:.6f}:")
print(f"Δ_A(ε*) = {Delta_A_at_star:.6f}")
print(f"Δ_B(ε*) = {Delta_B_at_star:.6f}")
print(f"Gap at ε*: {Delta_A_at_star - Delta_B_at_star:.6f} (should be ≈ 0)")
print("="*80)

**IMPORTANT NOTE:** The exact value ε* = 0.35106 must be computed using **exact unrounded** Δ values:
- Exact gap: 0.559375 - 0.580 = -0.020625
- Exact H diff: 0.258750 - 0.200 = 0.058750
- ε* = 0.020625 / 0.058750 = 0.35106

Using rounded values (gap ≈ 0.021, H_diff ≈ 0.059) gives ε* ≈ 0.356 - a **presentation trap** to avoid!

## 2. Gap Evolution Under Uniform Perturbation

Visualize how the gap Δ_A - Δ_B evolves as ε increases from 0 to 0.5.

In [ ]:
# Generate perturbation range
epsilon_range = np.linspace(0, 0.5, 100)

# Compute Δ values using analytical formula
Delta_A_curve = [analytical_uniform_perturbation(Delta_A_orig, H_A, eps) for eps in epsilon_range]
Delta_B_curve = [analytical_uniform_perturbation(Delta_B_orig, H_B, eps) for eps in epsilon_range]
Delta_C_curve = [analytical_uniform_perturbation(Delta_C_orig, H_C, eps) for eps in epsilon_range]

# Compute gap
gap_curve = [Delta_A_curve[i] - Delta_B_curve[i] for i in range(len(epsilon_range))]

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left panel: Δ values vs ε
ax1.plot(epsilon_range, Delta_A_curve, label='Δ_A (integrator)', linewidth=2)
ax1.plot(epsilon_range, Delta_B_curve, label='Δ_B (polymath)', linewidth=2)
ax1.plot(epsilon_range, Delta_C_curve, label='Δ_C (specialist)', linewidth=2)
ax1.axvline(epsilon_star, color='red', linestyle='--', linewidth=1.5, label=f'ε* = {epsilon_star:.3f}')
ax1.axhline(0, color='black', linestyle='-', linewidth=0.5)
ax1.set_xlabel('Perturbation ε', fontsize=12)
ax1.set_ylabel('Diversity Δ', fontsize=12)
ax1.set_title('Diversity Evolution Under Uniform Perturbation', fontsize=13)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Right panel: Gap Δ_A - Δ_B vs ε
ax2.plot(epsilon_range, gap_curve, linewidth=2, color='purple')
ax2.axhline(0, color='black', linestyle='-', linewidth=0.5)
ax2.axvline(epsilon_star, color='red', linestyle='--', linewidth=1.5, label=f'ε* = {epsilon_star:.3f}')
ax2.fill_between(epsilon_range, 0, gap_curve, where=[eps < epsilon_star for eps in epsilon_range], 
                 alpha=0.2, color='blue', label='Δ_A < Δ_B')
ax2.fill_between(epsilon_range, 0, gap_curve, where=[eps >= epsilon_star for eps in epsilon_range], 
                 alpha=0.2, color='orange', label='Δ_A > Δ_B')
ax2.set_xlabel('Perturbation ε', fontsize=12)
ax2.set_ylabel('Gap Δ_A - Δ_B', fontsize=12)
ax2.set_title('Gap Evolution and Inversion Point', fontsize=13)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nGap evolution summary:")
print(f"ε < {epsilon_star:.3f}: Δ_A < Δ_B, gap < {abs(gap_orig):.3f} (current state)")
print(f"ε = {epsilon_star:.3f}: gap = 0 (exact inversion)")
print(f"ε > {epsilon_star:.3f}: Δ_A > Δ_B, gap grows linearly")
print(f"\nIn all realistic scenarios (ε < 0.35), gap remains small (<0.02)")
print(f"relative to A/B-vs-C separation (~0.33).")

## 3. Non-Uniform Perturbation Scenarios

Test robustness under realistic perturbation patterns:

### Scenario 1: Neighbors Closer
Adjacent fields become more similar:
- $s_{ij} \to s_{ij} + 0.10$ for $|i-j| = 1$
- $s_{ij} \to s_{ij} - 0.05$ for $|i-j| \geq 2$

### Scenario 2: Uniform +0.10
All off-diagonal similarities increase uniformly:
- $s_{ij} \to s_{ij} + 0.10$ for all $i \neq j$

In [ ]:
def apply_neighbors_closer_perturbation(S_orig):
    """
    Scenario 1: Neighbors closer.
    
    s_ij → s_ij + 0.10 for |i-j| = 1
    s_ij → s_ij - 0.05 for |i-j| ≥ 2
    """
    S_pert = S_orig.copy()
    n = len(S_orig)
    
    for i in range(n):
        for j in range(n):
            if i == j:
                continue  # Keep diagonal = 1.0
            
            dist = abs(i - j)
            if dist == 1:
                S_pert[i, j] += 0.10
            else:
                S_pert[i, j] -= 0.05
    
    return S_pert

def apply_uniform_perturbation(S_orig, epsilon):
    """
    Scenario 2: Uniform perturbation.
    
    s_ij → s_ij + ε for all i ≠ j
    """
    S_pert = S_orig.copy()
    n = len(S_orig)
    
    for i in range(n):
        for j in range(n):
            if i != j:
                S_pert[i, j] += epsilon
    
    return S_pert

# Derive proportion vectors from publication data
p_A = pubs_A.sum(axis=0) / pubs_A.sum()
p_B = pubs_B.sum(axis=0) / pubs_B.sum()
p_C = pubs_C.sum(axis=0) / pubs_C.sum()

print("\n" + "="*80)
print("SCENARIO 1: NEIGHBORS CLOSER")
print("="*80)

S_scenario1 = apply_neighbors_closer_perturbation(S_orig)
print("\nPerturbed similarity matrix (Scenario 1):")
print(pd.DataFrame(S_scenario1, index=categories, columns=categories))

# Compute new Δ values
Delta_A_sc1 = compute_diversity_from_p(p_A, S_scenario1)
Delta_B_sc1 = compute_diversity_from_p(p_B, S_scenario1)
Delta_C_sc1 = compute_diversity_from_p(p_C, S_scenario1)

print("\nResults:")
print(f"{'Researcher':<15} {'Δ_orig':<12} {'Δ_pert':<12} {'Change':<12}")
print("-" * 60)
print(f"A (integrator)  {Delta_A_orig:<12.6f} {Delta_A_sc1:<12.6f} {Delta_A_sc1 - Delta_A_orig:<12.6f}")
print(f"B (polymath)    {Delta_B_orig:<12.6f} {Delta_B_sc1:<12.6f} {Delta_B_sc1 - Delta_B_orig:<12.6f}")
print(f"C (specialist)  {Delta_C_orig:<12.6f} {Delta_C_sc1:<12.6f} {Delta_C_sc1 - Delta_C_orig:<12.6f}")

gap_sc1 = abs(Delta_A_sc1 - Delta_B_sc1)
print(f"\nNew A-B gap: |Δ_A - Δ_B| = {gap_sc1:.6f} (was {abs(gap_orig):.6f})")
print(f"A/B-vs-C separation: {abs((Delta_A_sc1 + Delta_B_sc1)/2 - Delta_C_sc1):.3f}")
print("\nVerdict: Discrimination preserved; A ≈ B remains; C well separated.")
print("="*80)

In [ ]:
print("\n" + "="*80)
print("SCENARIO 2: UNIFORM +0.10")
print("="*80)

epsilon_sc2 = 0.10
S_scenario2 = apply_uniform_perturbation(S_orig, epsilon_sc2)

print(f"\nPerturbation: s_ij → s_ij + {epsilon_sc2} for all i ≠ j")
print("\nPerturbed similarity matrix (Scenario 2):")
print(pd.DataFrame(S_scenario2, index=categories, columns=categories))

# Compute new Δ values using analytical formula
Delta_A_sc2_analytical = analytical_uniform_perturbation(Delta_A_orig, H_A, epsilon_sc2)
Delta_B_sc2_analytical = analytical_uniform_perturbation(Delta_B_orig, H_B, epsilon_sc2)
Delta_C_sc2_analytical = analytical_uniform_perturbation(Delta_C_orig, H_C, epsilon_sc2)

# Also compute directly from perturbed matrix (verification)
Delta_A_sc2_direct = compute_diversity_from_p(p_A, S_scenario2)
Delta_B_sc2_direct = compute_diversity_from_p(p_B, S_scenario2)
Delta_C_sc2_direct = compute_diversity_from_p(p_C, S_scenario2)

print("\nResults (using analytical formula):")
print(f"{'Researcher':<15} {'Δ_orig':<12} {'ε(1-H)':<12} {'Δ_pert':<12}")
print("-" * 60)
print(f"A (integrator)  {Delta_A_orig:<12.6f} {epsilon_sc2*(1-H_A):<12.6f} {Delta_A_sc2_analytical:<12.6f}")
print(f"B (polymath)    {Delta_B_orig:<12.6f} {epsilon_sc2*(1-H_B):<12.6f} {Delta_B_sc2_analytical:<12.6f}")
print(f"C (specialist)  {Delta_C_orig:<12.6f} {epsilon_sc2*(1-H_C):<12.6f} {Delta_C_sc2_analytical:<12.6f}")

print("\nVerification (direct computation from perturbed matrix):")
print(f"Δ_A (direct): {Delta_A_sc2_direct:.6f} vs analytical: {Delta_A_sc2_analytical:.6f}")
print(f"Δ_B (direct): {Delta_B_sc2_direct:.6f} vs analytical: {Delta_B_sc2_analytical:.6f}")
print(f"Δ_C (direct): {Delta_C_sc2_direct:.6f} vs analytical: {Delta_C_sc2_analytical:.6f}")

gap_sc2 = abs(Delta_A_sc2_analytical - Delta_B_sc2_analytical)
print(f"\nNew A-B gap: |Δ_A - Δ_B| = {gap_sc2:.6f} (was {abs(gap_orig):.6f})")
print(f"A/B-vs-C separation: {abs((Delta_A_sc2_analytical + Delta_B_sc2_analytical)/2 - Delta_C_sc2_analytical):.3f}")
print("\nVerdict: Discrimination preserved.")
print("="*80)

## 4. Invariance of S and E Under Similarity Perturbation

**Key insight:** Only Δ is sensitive to the similarity matrix s_ij. The other panel components are invariant:

### S (Coherence) Invariance

$$S = \frac{1}{\binom{n}{2}} \sum_{k < l} \cos(\mathbf{r}_k, \mathbf{r}_l)$$

S depends only on publication reference vectors $\mathbf{r}_k$, NOT on category-level similarity s_ij.

**S is invariant under perturbation of s_ij.**

### E (Cross-Field Effect) Invariance

$$E = \frac{\text{citations from outside primary category}}{\text{total citations}}$$

E depends on citation patterns and primary category assignments (determined by reference vectors), NOT on s_ij.

**E is invariant under perturbation of s_ij.**

In [ ]:
# Original S and E values (from panel-worked-example)
S_A = 0.733
S_B = 0.000
S_C = 0.881

E_A = 0.600
E_B = 0.063
E_C = 0.211

print("\n" + "="*80)
print("INVARIANCE OF S AND E UNDER SIMILARITY PERTURBATION")
print("="*80)

print("\nCoherence (S) values:")
print(f"{'Researcher':<15} {'S (original)':<15} {'S (any pert.)':<15} {'Status':<15}")
print("-" * 60)
print(f"A (integrator)  {S_A:<15.3f} {S_A:<15.3f} INVARIANT")
print(f"B (polymath)    {S_B:<15.3f} {S_B:<15.3f} INVARIANT")
print(f"C (specialist)  {S_C:<15.3f} {S_C:<15.3f} INVARIANT")

print("\nCross-field effect (E) values:")
print(f"{'Researcher':<15} {'E (original)':<15} {'E (any pert.)':<15} {'Status':<15}")
print("-" * 60)
print(f"A (integrator)  {E_A:<15.3f} {E_A:<15.3f} INVARIANT")
print(f"B (polymath)    {E_B:<15.3f} {E_B:<15.3f} INVARIANT")
print(f"C (specialist)  {E_C:<15.3f} {E_C:<15.3f} INVARIANT")

print("\nExplanation:")
print("- S measures bibliographic coupling between publication reference vectors")
print("  → Function of r_k vectors only, independent of s_ij")
print("- E measures citation patterns across category boundaries")
print("  → Function of citation data and primary categories, independent of s_ij")
print("- Only Δ uses the similarity matrix s_ij in its computation")
print("  → Δ is sensitive to perturbations, but changes are predictable and small")
print("="*80)

## 5. Robustness Summary Table

Compare panel discrimination across all scenarios:

In [ ]:
# Create comprehensive summary
summary = pd.DataFrame({
    'Scenario': [
        'Original',
        'Neighbors closer',
        'Uniform +0.10'
    ],
    '|Δ_A - Δ_B|': [
        abs(gap_orig),
        gap_sc1,
        gap_sc2
    ],
    '|Δ_{A,B} - Δ_C|': [
        abs((Delta_A_orig + Delta_B_orig)/2 - Delta_C_orig),
        abs((Delta_A_sc1 + Delta_B_sc1)/2 - Delta_C_sc1),
        abs((Delta_A_sc2_analytical + Delta_B_sc2_analytical)/2 - Delta_C_sc2_analytical)
    ],
    'Panel discrimination': [
        'Full (all 3 types separated)',
        'Full',
        'Full'
    ]
})

print("\n" + "="*80)
print("ROBUSTNESS SUMMARY")
print("="*80)
print(summary.to_string(index=False))
print("="*80)

print("\n**KEY FINDINGS:**")
print("\n1. Small A-B gap is structurally robust:")
print(f"   - Gap remains in range [{gap_sc1:.3f}, {abs(gap_orig):.3f}] across realistic perturbations")
print(f"   - Requires ε* = {epsilon_star:.3f} (35% perturbation) to achieve exact inversion")

print("\n2. A/B-vs-C separation is large and stable:")
print(f"   - Separation remains ~0.30-0.35 across all scenarios")
print(f"   - Panel clearly distinguishes specialists from broad researchers")

print("\n3. S and E provide robustness:")
print("   - Both invariant under s_ij perturbations")
print("   - Discriminate A from B regardless of similarity matrix choice")
print("   - Multi-component panel is more robust than single-scalar Δ")

print("\n4. Analytical formula enables predictability:")
print("   - Δ changes are deterministic: Δ_new = Δ_old - ε(1 - H)")
print("   - No surprises or threshold effects below ε*")
print("="*80)

## Conclusion

This sensitivity analysis demonstrates that the panel approach is **structurally robust**:

1. The small gap Δ_A ≈ Δ_B is not an artifact - it requires 35% perturbation (ε* = 0.351) to invert
2. In all realistic perturbation scenarios, panel discrimination is preserved
3. S and E are invariant under similarity matrix changes, providing robustness
4. The analytical formula Δ_new = Δ_old - ε(1 - H) enables predictable behavior

**Implication for evaluation practice:** The panel (Δ, S, E) can be used with confidence. Small variations in category similarity definitions will not qualitatively change the discrimination between integrators, polymaths, and specialists.